In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week9-lesson-7"). \
config("spark.sql.warehouse.dir", f"/user/itv027484/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

## How spark determines initial number of partitions of a DF on reading a folder with multiple file

### read a 1 gb file, repartition it, write it, then read it back. check number of partitions

In [2]:
order_schema= 'order_id long, order_date string , customer_id long, order_status string'

In [3]:
orders_df = spark.read.format('csv').schema(order_schema).load('/public/trendytech/orders/orders_1gb.csv')

In [4]:
orders_df.rdd.getNumPartitions()

9

In [5]:
orders_df_parts = orders_df.repartition(20)

In [6]:
orders_df_parts.rdd.getNumPartitions()

20

In [7]:
orders_df_parts.write.format('csv').mode('overwrite').save('orders_20p_csv')

In [8]:
new_df = spark.read.format('csv').schema(order_schema).load('orders_20p_csv')

In [9]:
new_df.rdd.getNumPartitions()

10

In [10]:
spark.conf.get('spark.sql.files.openCostInBytes')  ## 4MB  - time talen to open 1 file

'4194304'

In [11]:
spark.conf.get("spark.sql.files.maxPartitionBytes") ## 128MB

'134217728b'

In [12]:
spark.sparkContext.defaultParallelism  ## based on number of cores available

10

### How Spark Calculates Partitions on Read - multiple files in a folder
### When Spark calculates file splits for a FileScan, it computes the target split size using this general formula
### Split Size = min(maxPartitionBytes, max(openCostInBytes,Bytes Per Core))

In [13]:
## small file problem - too many small files degrades performance

Concrete Example
Suppose you read a folder containing 40 Parquet files, each 10 MB in size, on a cluster with:
spark.sql.files.maxPartitionBytes = 128 MB
spark.sql.files.openCostInBytes = 4 MB
spark.default.parallelism = 8 cores  

Total Effective Size:
$$40 \times (10\text{ MB} + 4\text{ MB}) = 560\text{ MB}
$$Bytes Per Core:
$$\frac{560\text{ MB}}{8\text{ cores}} = 70\text{ MB}
$$Target Split Size:
$$\min(128\text{ MB}, \max(4\text{ MB}, 70\text{ MB})) = 70\text{ MB}
$$Files Packed per Partition:Each 10 MB file has an effective size of 14 MB.
$$\frac{70\text{ MB}}{14\text{ MB}} = 5\text{ files per partition}
$$Final Partition Count:
$$\frac{40\text{ files}}{5\text{ files/partition}} = \mathbf{8\text{ Partitions}}
$$Result: Spark creates exactly 8 partitions, assigning 5 small files to each partition, perfectly utilizing all 8 CPU cores.

In [14]:
orders_df_parts1 = orders_df.repartition(500)

In [16]:
orders_df_parts1.rdd.getNumPartitions()

500

In [20]:
orders_df_parts1.write.format('csv').mode('overwrite').save('orders_500p_csv')

In [21]:
new_df1 = spark.read.format('csv').schema(order_schema).load('orders_500p_csv')

In [22]:
new_df1.rdd.getNumPartitions()

24

In [27]:
## 1 GB file - split to 500 parts. approx 2 mb per part. 
## But on reading, total effective size of 500 files, spark considers = 500 * ( 2mb + 4mb(i.e maxPartitionBytes)) ~ approx 3gb
## To create 128mb partitions, we need around ( 128 / 6 = ) approx 21 files.
## so 500 files -> combibned to  21 files = 24 partitions will be created

#### paritition Size = min(maxPartitionBytes, max(openCostInBytes,Bytes Per Core))

In [29]:
 # i.e min (128mb, max(4mb, 1gb/2cores i.e 500mb ) )  = 128mb

In [30]:
## if we had more cores available, then we would have more number of partitions

In [31]:
### Spark always tries to utilise all the available cores for max parallelism